# Iris Dataset - K Nearest Neighbours

This practices k nearest neighbours with train test and validation splits

## Notebook setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

## Iris dataset analysis

### Read and exploratory

In [ ]:
iris = load_iris()
df_iris = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df_iris

In [ ]:
# Check for missing values and data types
df_iris.info()
# No missing values, all features are numeric except for species

In [ ]:
# Check to see if normalisation is needed
df_iris.describe()
# The features are on different scales, so we will need to normalise the data before doing anything "fun"

In [ ]:
# pairs plots without plotting species
plt.figure(figsize=(10, 10))
pd.plotting.scatter_matrix(df_iris, figsize=(10, 10), diagonal='kde')
plt.suptitle('Pairs Plot of Scaled Iris Dataset', fontsize=16)
plt.show()


### K nearest neighbours
Using a train test split - remember to split before scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_iris, iris.target, test_size=0.2, random_state=123) 
#df_iris has only numeric, iris has the target names (species), so we can use that for y

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

k_values = [1, 3, 5, 7, 11, 15]

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    y_pred = knn.predict(X_test_scaled)
    score = knn.score(X_test_scaled, y_test)
    
    print(f"k={k}, Test Accuracy: {score:.3f}")

### Adding in validation

In [ ]:
# split out train into train and validation sets
X_train_val, X_test, y_train_val, y_test = train_test_split(df_iris, iris.target, test_size=0.2, stratify=iris.target, random_state=123) 

X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, random_state=123, stratify=y_train_val)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

k_values = [1, 3, 5, 7, 11, 15]

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    y_pred = knn.predict(X_val_scaled)
    score = knn.score(X_val_scaled, y_val)
    
    print(f"k={k}, Validation Accuracy: {score:.3f}")

Many k values report 96% accuracy. Let's use 5.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
    
y_pred = knn.predict(X_test_scaled)
score = knn.score(X_test_scaled, y_test)

print(f"k=5, Test Accuracy: {score:.3f}")

### Incorporate KFold cross-validation

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

X_train_val, X_test, y_train_val, y_test = train_test_split(df_iris, iris.target, test_size=0.2, stratify=iris.target, random_state=123) 

skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
k_values = [1, 3, 5, 7, 11, 15]

for k in k_values:
    fold_accuracies = []

    for train_idx, val_idx in skf5.split(X_train_val, y_train_val):
        # Use .iloc for row selection
        X_train = X_train_val.iloc[train_idx]
        X_val = X_train_val.iloc[val_idx]
        y_train = y_train_val[train_idx]
        y_val = y_train_val[val_idx]

        # Scale inside each fold
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        # Train the model
        model = KNeighborsClassifier(n_neighbors=k)
        model.fit(X_train_scaled, y_train)

        # Predict + evaluate
        y_pred = model.predict(X_val_scaled)
        fold_accuracies.append(accuracy_score(y_val, y_pred))

    print(f"k={k}, Mean cross validation accuracy:", np.mean(fold_accuracies))

Using stratified K fold cross validation, k=3 looks good again 

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_val)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_scaled, y_train_val)
    
y_pred = knn.predict(X_test_scaled)
score = knn.score(X_test_scaled, y_test)

print(f"k=3, Test Accuracy: {score:.3f}")

My final model performs at 93.3% accuracy on the test set for identifying species